# NutriDermAI — RAG Corpus Builder
## Stage 7: ChromaDB Vector Index for Dermatology VQA

**Run order:** Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9

| Cell | Purpose |
|------|---------|
| 1 | Install packages |
| 2 | Config + folder setup |
| 3 | Write 26 hardcoded dermatology QA pairs (no internet needed) |
| 4 | Chunker utility |
| 5 | Indexer utility |
| 6 | Build RAG index (main cell) |
| 7 | Test retrieval — all question types |
| 8 | Stats + bot readiness check |
| 9 | Update rag_config.py paths |

**Corpus folder layout:**
```
/home/vjti-comp/Desktop/Final Project Code/VQA/rag_corpus/
  statpearls/       <- paste text from ncbi.nlm.nih.gov/books/NBK519567/ etc.
  abcde_guidelines/ <- paste text from aad.org ABCDE page
  aad_guidelines/   <- paste text from aad.org treatment pages
  dermnet/          <- paste text from dermnetnz.org pages
  medquad/          <- auto-written by Cell 3 (no paste needed)
```


In [1]:
# Cell 1: Install packages
import subprocess, sys

pkgs = [
    "chromadb>=0.4.0",
    "sentence-transformers>=2.2.0",
    "tqdm",
]
for pkg in pkgs:
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", pkg],
        capture_output=True, text=True)
    print(f"  [{'OK' if r.returncode==0 else 'FAIL'}] {pkg}")

print("\nDone. Proceed to Cell 2.")


  [OK] chromadb>=0.4.0
  [OK] sentence-transformers>=2.2.0
  [OK] tqdm

Done. Proceed to Cell 2.


In [2]:
# Cell 2: Config + Folder Setup
import os, sys, re, uuid, json
from pathlib import Path
from collections import Counter
from tqdm import tqdm

# ── Paths — RTX 4070 machine ──────────────────────────────────────────────
PROJECT_DIR    = Path("/home/vjti-comp/Desktop/Final Project Code/VQA")
RAG_CORPUS_DIR = PROJECT_DIR / "rag_corpus"
CHROMA_DB_DIR  = PROJECT_DIR / "chroma_db"

# Must match rag_config.py exactly
CHROMA_COLLECTION_NAME = "dermatology_kb"
EMBEDDING_MODEL        = "all-MiniLM-L6-v2"
CHUNK_SIZE             = 350
CHUNK_OVERLAP          = 60
MIN_CHUNK_CHARS        = 40
TOP_K                  = 3
SIMILARITY_THRESHOLD   = 0.25
EMBED_BATCH_SIZE       = 64

# Create folder structure
for sub in ["statpearls", "abcde_guidelines", "aad_guidelines", "dermnet", "medquad"]:
    (RAG_CORPUS_DIR / sub).mkdir(parents=True, exist_ok=True)
CHROMA_DB_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print(" Config")
print("=" * 60)
print(f"  Project    : {PROJECT_DIR}")
print(f"  RAG corpus : {RAG_CORPUS_DIR}")
print(f"  ChromaDB   : {CHROMA_DB_DIR}")
print(f"  Model      : {EMBEDDING_MODEL}")
print(f"  Chunk size : {CHUNK_SIZE} words | overlap={CHUNK_OVERLAP}")
print()

# Check existing documents
docs = [f for f in RAG_CORPUS_DIR.rglob("*.txt") if f.is_file()]
if docs:
    print(f"  Existing documents: {len(docs)}")
    for src, cnt in sorted(Counter(f.parent.name for f in docs).items()):
        size_kb = sum(f.stat().st_size for f in docs if f.parent.name == src) // 1024
        print(f"    {src:<25} {cnt:>3} files  {size_kb:>5} KB")
else:
    print("  No documents yet — Cell 3 will write the medquad/ QA pairs.")
    print("  For statpearls/, aad_guidelines/, dermnet/, abcde_guidelines/:")
    print("  Open each URL in browser -> Ctrl+A -> Ctrl+C -> paste into .txt file")
    print()
    print("  URLs:")
    urls = [
        ("statpearls/melanoma.txt",           "https://www.ncbi.nlm.nih.gov/books/NBK519567/"),
        ("statpearls/bcc.txt",                "https://www.ncbi.nlm.nih.gov/books/NBK482439/"),
        ("statpearls/scc.txt",                "https://www.ncbi.nlm.nih.gov/books/NBK441939/"),
        ("statpearls/actinic_keratosis.txt",  "https://www.ncbi.nlm.nih.gov/books/NBK557401/"),
        ("statpearls/seborrheic_k.txt",       "https://www.ncbi.nlm.nih.gov/books/NBK545285/"),
        ("statpearls/nevus.txt",              "https://www.ncbi.nlm.nih.gov/books/NBK459260/"),
        ("abcde_guidelines/aad_abcde.txt",    "https://www.aad.org/public/diseases/skin-cancer/find/at-risk/abcdes"),
        ("aad_guidelines/melanoma.txt",       "https://www.aad.org/public/diseases/skin-cancer/types/common/melanoma"),
        ("aad_guidelines/bcc.txt",            "https://www.aad.org/public/diseases/skin-cancer/types/common/bcc"),
        ("aad_guidelines/scc.txt",            "https://www.aad.org/public/diseases/skin-cancer/types/common/scc"),
        ("aad_guidelines/actinic_k.txt",      "https://www.aad.org/public/diseases/a-z/actinic-keratosis-overview"),
        ("dermnet/melanoma.txt",              "https://dermnetnz.org/topics/melanoma"),
        ("dermnet/bcc.txt",                   "https://dermnetnz.org/topics/basal-cell-carcinoma"),
        ("dermnet/scc.txt",                   "https://dermnetnz.org/topics/squamous-cell-carcinoma"),
        ("dermnet/actinic_keratosis.txt",     "https://dermnetnz.org/topics/actinic-keratosis"),
        ("dermnet/seborrheic_k.txt",          "https://dermnetnz.org/topics/seborrhoeic-keratosis"),
        ("dermnet/nevus.txt",                 "https://dermnetnz.org/topics/melanocytic-naevus"),
    ]
    for fname, url in urls:
        print(f"    {fname:<40} <- {url}")

print("\nReady. Run Cell 3 next.")


 Config
  Project    : /home/vjti-comp/Desktop/Final Project Code/VQA
  RAG corpus : /home/vjti-comp/Desktop/Final Project Code/VQA/rag_corpus
  ChromaDB   : /home/vjti-comp/Desktop/Final Project Code/VQA/chroma_db
  Model      : all-MiniLM-L6-v2
  Chunk size : 350 words | overlap=60

  No documents yet — Cell 3 will write the medquad/ QA pairs.
  For statpearls/, aad_guidelines/, dermnet/, abcde_guidelines/:
  Open each URL in browser -> Ctrl+A -> Ctrl+C -> paste into .txt file

  URLs:
    statpearls/melanoma.txt                  <- https://www.ncbi.nlm.nih.gov/books/NBK519567/
    statpearls/bcc.txt                       <- https://www.ncbi.nlm.nih.gov/books/NBK482439/
    statpearls/scc.txt                       <- https://www.ncbi.nlm.nih.gov/books/NBK441939/
    statpearls/actinic_keratosis.txt         <- https://www.ncbi.nlm.nih.gov/books/NBK557401/
    statpearls/seborrheic_k.txt              <- https://www.ncbi.nlm.nih.gov/books/NBK545285/
    statpearls/nevus.txt             

In [3]:
# Cell 3b: AAD HTML Parser
# AAD pages are HTML — save the page as .html from your browser,
# then run this cell to extract clean text and save to corpus.
#
# How to save AAD pages:
#   1. Open the URL in Chrome/Firefox
#   2. Wait for page to fully load
#   3. Ctrl+S -> Save as "Webpage, Complete" or "Webpage, HTML Only"
#   4. Move the saved .html file to one of the INPUT_HTML_DIR folders below
#   5. Run this cell

import re
from pathlib import Path

try:
    from bs4 import BeautifulSoup
    print("[OK] beautifulsoup4 available")
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "beautifulsoup4", "lxml"],
                   capture_output=True)
    from bs4 import BeautifulSoup
    print("[OK] beautifulsoup4 installed")

# ── Where to look for saved HTML files ───────────────────────────────────
# Put your saved .html files anywhere under these folders
# The script will find them automatically
HTML_SEARCH_DIRS = [
    PROJECT_DIR / "html_input",
    PROJECT_DIR / "aad_html",
    Path.home() / "Downloads",
    Path("/tmp"),
]

# ── Mapping: HTML filename keywords -> corpus output folder + filename ────
# If the saved HTML filename contains the keyword, it goes to that output
HTML_ROUTING = [
    # AAD pages -> aad_guidelines/
    ("melanoma",        "aad_guidelines", "melanoma.txt"),
    ("bcc",             "aad_guidelines", "bcc.txt"),
    ("basal",           "aad_guidelines", "bcc.txt"),
    ("scc",             "aad_guidelines", "scc.txt"),
    ("squamous",        "aad_guidelines", "scc.txt"),
    ("actinic",         "aad_guidelines", "actinic_keratosis.txt"),
    ("abcde",           "abcde_guidelines", "aad_abcde.txt"),
    ("skin-cancer",     "aad_guidelines", "skin_cancer.txt"),
    ("prevention",      "aad_guidelines", "prevention.txt"),
    # DermNet pages -> dermnet/
    ("dermnet",         "dermnet",        None),   # None = use original stem
    ("dermnetnz",       "dermnet",        None),
    # StatPearls pages -> statpearls/
    ("nbk",             "statpearls",     None),
    ("statpearls",      "statpearls",     None),
]

def clean_html(html_path):
    """Extract clean readable text from a saved HTML file."""
    raw = Path(html_path).read_text(encoding="utf-8", errors="ignore")
    soup = BeautifulSoup(raw, "lxml")

    # Remove non-content tags
    for tag in soup(["script", "style", "nav", "footer", "header",
                     "aside", "form", "button", "iframe", "noscript",
                     "svg", "img", "figure", "figcaption", "meta",
                     "link", "head"]):
        tag.decompose()

    # Try to find main content area
    main = (
        soup.find("main") or
        soup.find("article") or
        soup.find(attrs={"role": "main"}) or
        soup.find("div", class_=re.compile(r"content|article|main|body", re.I)) or
        soup.find("div", id=re.compile(r"content|main|article", re.I)) or
        soup.body or soup
    )

    # Get text with newlines between block elements
    lines = []
    for elem in main.descendants:
        if hasattr(elem, "get_text"):
            continue
        text = str(elem).strip()
        if text and len(text) > 2:
            lines.append(text)

    # Simpler approach — just get all text
    text = main.get_text(separator=" ", strip=True)

    # Clean up whitespace and repeated characters
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    text = re.sub(r"(.)\1{4,}", r"\1", text)  # remove "........." runs
    text = re.sub(r"[|]{2,}", "", text)           # remove "|||" runs
    text = text.strip()

    return text

def route_html(html_path):
    """Determine output folder and filename for an HTML file."""
    stem = html_path.stem.lower().replace("-", "_").replace(" ", "_")
    for keyword, folder, fname in HTML_ROUTING:
        if keyword in stem or keyword in str(html_path).lower():
            out_fname = fname if fname else f"{stem}.txt"
            return folder, out_fname
    # Default: put in aad_guidelines if from aad.org, else dermnet
    if "aad" in stem:
        return "aad_guidelines", f"{stem}.txt"
    return "dermnet", f"{stem}.txt"

# ── Find all HTML files ───────────────────────────────────────────────────
found_html = []
for search_dir in HTML_SEARCH_DIRS:
    if search_dir.exists():
        for ext in ["*.html", "*.htm"]:
            found_html.extend(search_dir.glob(ext))

print(f"Found {len(found_html)} HTML files:\n")

if not found_html:
    print("No HTML files found. Save AAD pages from browser first:")
    print()
    aad_urls = [
        ("aad_guidelines/melanoma.txt",
         "https://www.aad.org/public/diseases/skin-cancer/types/common/melanoma"),
        ("aad_guidelines/bcc.txt",
         "https://www.aad.org/public/diseases/skin-cancer/types/common/bcc"),
        ("aad_guidelines/scc.txt",
         "https://www.aad.org/public/diseases/skin-cancer/types/common/scc"),
        ("aad_guidelines/actinic_keratosis.txt",
         "https://www.aad.org/public/diseases/a-z/actinic-keratosis-overview"),
        ("abcde_guidelines/aad_abcde.txt",
         "https://www.aad.org/public/diseases/skin-cancer/find/at-risk/abcdes"),
    ]
    for out, url in aad_urls:
        print(f"  {out}")
        print(f"    <- {url}")
        print(f"    Open in browser -> Ctrl+S -> Save HTML -> put in {PROJECT_DIR}/html_input/")
        print()
    print(f"Then re-run this cell.")
else:
    converted = 0
    skipped   = 0
    for html_path in found_html:
        try:
            text = clean_html(html_path)
            if len(text) < 200:
                print(f"  [SKIP] {html_path.name} — too short ({len(text)} chars)")
                skipped += 1
                continue

            folder, fname = route_html(html_path)
            out_path = RAG_CORPUS_DIR / folder / fname
            out_path.parent.mkdir(parents=True, exist_ok=True)
            out_path.write_text(text, encoding="utf-8")
            converted += 1
            print(f"  [OK] {html_path.name}")
            print(f"       -> {folder}/{fname}  ({len(text):,} chars)")
        except Exception as e:
            print(f"  [FAIL] {html_path.name}: {e}")
            skipped += 1

    print(f"\nConverted: {converted}  Skipped: {skipped}")
    print()
    print("Corpus now:")
    for sub in ["statpearls","abcde_guidelines","aad_guidelines","dermnet","medquad"]:
        cnt = len(list((RAG_CORPUS_DIR / sub).glob("*.txt")))
        tag = "[OK]" if cnt > 0 else "[EMPTY]"
        print(f"  {tag} {sub:<25} {cnt} files")
    print()
    if converted > 0:
        print("Re-run Cell 6 with RESET=True to rebuild the index with new files.")


[OK] beautifulsoup4 available
Found 0 HTML files:

No HTML files found. Save AAD pages from browser first:

  aad_guidelines/melanoma.txt
    <- https://www.aad.org/public/diseases/skin-cancer/types/common/melanoma
    Open in browser -> Ctrl+S -> Save HTML -> put in /home/vjti-comp/Desktop/Final Project Code/VQA/html_input/

  aad_guidelines/bcc.txt
    <- https://www.aad.org/public/diseases/skin-cancer/types/common/bcc
    Open in browser -> Ctrl+S -> Save HTML -> put in /home/vjti-comp/Desktop/Final Project Code/VQA/html_input/

  aad_guidelines/scc.txt
    <- https://www.aad.org/public/diseases/skin-cancer/types/common/scc
    Open in browser -> Ctrl+S -> Save HTML -> put in /home/vjti-comp/Desktop/Final Project Code/VQA/html_input/

  aad_guidelines/actinic_keratosis.txt
    <- https://www.aad.org/public/diseases/a-z/actinic-keratosis-overview
    Open in browser -> Ctrl+S -> Save HTML -> put in /home/vjti-comp/Desktop/Final Project Code/VQA/html_input/

  abcde_guidelines/aad_abc

In [4]:
# Cell 3: Write 26 Hardcoded Dermatology QA Pairs
# No internet needed — writes directly to rag_corpus/medquad/
# These cover all 6 disease classes + ABCDE + treatment + risk

DERM_QA = [
    # ── Melanoma ─────────────────────────────────────────────────────────
    ("What is melanoma?",
     "Melanoma is a malignant tumour of melanocytes, the pigment-producing cells of the "
     "skin. It is the most lethal form of skin cancer, responsible for over 57,000 deaths "
     "annually worldwide. Risk factors include UV exposure, fair skin (Fitzpatrick I-II), "
     "family history, BRAF/NRAS mutations, and the presence of multiple atypical nevi."),

    ("What are the ABCDE criteria for melanoma?",
     "ABCDE criteria: Asymmetry (one half does not mirror the other), Border irregularity "
     "(ragged, notched, or blurred edges), Color variation (multiple shades of brown, black, "
     "red, white, or blue-grey within one lesion), Diameter greater than 6mm, Evolution "
     "(any change in size, shape, color, or new symptoms such as bleeding or itching)."),

    ("How is melanoma treated?",
     "Early melanoma is treated with wide local excision with margins 0.5-2.0cm depending "
     "on Breslow thickness. Melanoma over 0.8mm requires sentinel lymph node biopsy. Advanced "
     "melanoma is treated with immunotherapy (pembrolizumab, nivolumab, ipilimumab) or targeted "
     "therapy (vemurafenib + dabrafenib for BRAF V600 mutations). 5-year survival: Stage I >98%, "
     "Stage IV 15-20% with modern immunotherapy."),

    ("What surgical margins are recommended for melanoma?",
     "Margins by Breslow thickness: in situ 0.5-1.0cm; up to 1.0mm requires 1.0cm margins; "
     "1.01-2.0mm requires 1-2cm margins; over 2.0mm requires 2.0cm margins. Sentinel lymph "
     "node biopsy is standard for melanomas >0.8mm or those with ulceration or high mitotic rate."),

    ("What is the prognosis for melanoma?",
     "Five-year survival by stage: Stage I (localized thin) >98%, Stage II (thick/ulcerated) "
     "65-90%, Stage III (regional lymph nodes) 40-78%, Stage IV (distant metastasis) 15-20%. "
     "Breslow thickness, ulceration, and mitotic rate are the most important prognostic factors."),

    # ── ABCDE / Dermoscopy ────────────────────────────────────────────────
    ("What is the ABCD rule of dermoscopy?",
     "The dermoscopic ABCD rule scores: Asymmetry 0-2 points (two perpendicular axes), "
     "Border abruptness 0-8 points (8 segments), Color richness 1-6 points (up to 6 "
     "dermoscopic colors), Dermoscopic structures 1-5 points (network, dots, streaks, "
     "regression). Total Dermoscopic Score = (A×1.3)+(B×0.1)+(C×0.5)+(D×0.5). "
     "TDS <4.75 = benign, 4.75-5.45 = suspicious, >5.45 = highly suspicious for melanoma."),

    ("What does asymmetry mean in dermoscopy?",
     "Asymmetry is evaluated by bisecting the lesion along two perpendicular axes through "
     "its geometric center. Score 0 = symmetric on both axes; score 1 = asymmetric on one "
     "axis; score 2 = asymmetric on both axes. An asymmetry score of 2 contributes 2.6 points "
     "to the Total Dermoscopic Score and is a strong predictor of melanoma."),

    ("What color variations indicate malignancy in skin lesions?",
     "Presence of 5 or 6 dermoscopic colors is strongly associated with melanoma. The six "
     "evaluated colors are: light brown, dark brown, black (superficial or deep melanin), "
     "red (vascularity or inflammation), white (regression or fibrosis), and blue-grey "
     "(melanin in dermis). Benign lesions typically show only 1-2 colors."),

    ("What does border irregularity indicate in skin lesion analysis?",
     "Irregular, poorly defined, or notched borders suggest malignancy. In the dermoscopic "
     "ABCD rule, the border is divided into 8 pie-slice segments. Each segment with abrupt "
     "pigment network cutoff scores 1 point. Ragged edges with satellite pigmentation are "
     "characteristic of melanoma."),

    # ── Basal Cell Carcinoma ──────────────────────────────────────────────
    ("What is basal cell carcinoma?",
     "Basal cell carcinoma is the most common skin cancer (80% of non-melanoma cases), "
     "arising from basal keratinocytes. Subtypes include nodular, superficial, morpheaform, "
     "and pigmented BCC. It rarely metastasises but causes significant local tissue destruction. "
     "Most common on sun-exposed areas of the head and neck."),

    ("How is basal cell carcinoma treated?",
     "BCC treatments: surgical excision (95% cure rate), Mohs micrographic surgery (98-99% "
     "cure rate for high-risk/recurrent BCC), electrodesiccation and curettage for small "
     "superficial lesions, cryotherapy, topical imiquimod or 5-fluorouracil for superficial BCC, "
     "radiation therapy, and vismodegib/sonidegib (hedgehog pathway inhibitors) for locally "
     "advanced or metastatic BCC."),

    # ── Squamous Cell Carcinoma ───────────────────────────────────────────
    ("What is squamous cell carcinoma of the skin?",
     "Cutaneous SCC arises from epidermal keratinocytes and is the second most common skin "
     "cancer. Unlike BCC, SCC has significant metastatic potential, especially in high-risk "
     "locations (ear, lip, temple) and immunocompromised patients. Can arise from actinic "
     "keratoses, chronic wounds, or de novo. High-risk features: diameter >2cm, depth >2mm, "
     "poor differentiation, perineural invasion."),

    ("How is squamous cell carcinoma treated?",
     "Low-risk SCC: excision (4-6mm margins) or curettage and electrodesiccation. "
     "High-risk SCC: Mohs surgery or wide excision (6-10mm margins), sentinel lymph node "
     "biopsy consideration, adjuvant radiation. Locally advanced or metastatic SCC: "
     "cemiplimab (anti-PD-1) or pembrolizumab immunotherapy; platinum-based chemotherapy "
     "as alternative."),

    # ── Actinic Keratosis ─────────────────────────────────────────────────
    ("What is actinic keratosis?",
     "Actinic keratosis (AK) is a common premalignant epithelial lesion from cumulative UV "
     "exposure. Presents as rough, scaly patches on sun-damaged skin. Histologically shows "
     "atypical keratinocyte proliferation in the lower epidermis. Annual risk of individual "
     "AK progressing to invasive SCC is 0.1-0.5%. Treatment: cryotherapy, topical "
     "5-fluorouracil, imiquimod, ingenol mebutate, photodynamic therapy, diclofenac gel."),

    ("Is actinic keratosis dangerous?",
     "AK itself is not invasive cancer but is a precancerous condition. Approximately 5-10% "
     "of untreated AKs may progress to squamous cell carcinoma. Patients with multiple AKs "
     "have substantially higher cumulative risk. Treatment is recommended to prevent progression. "
     "Multiple AKs indicate significant cumulative sun damage and increased skin cancer risk."),

    # ── Seborrheic Keratosis ──────────────────────────────────────────────
    ("What is seborrheic keratosis?",
     "Seborrheic keratosis is the most common benign epithelial tumour, appearing as waxy, "
     "stuck-on plaques from tan to dark brown. No malignant potential. Treatment only if "
     "irritated or cosmetically bothersome. Dermoscopic features: comedo-like openings, "
     "milia-like cysts, fissures and ridges (brain-like pattern), hairpin vessels. "
     "Treatment: cryotherapy, curettage, electrodesiccation, laser ablation."),

    ("How is seborrheic keratosis distinguished from melanoma?",
     "SK has a stuck-on waxy appearance with well-defined borders and warty surface. "
     "Dermoscopically SK shows milia-like cysts, comedo-like openings, and sharp borders. "
     "Melanoma shows atypical pigment network, regression structures, and irregular vascular "
     "patterns. ABCDE criteria are not typically positive for SK. When uncertain, "
     "dermoscopy or excision and histopathology are used to confirm."),

    # ── Melanocytic Nevus ─────────────────────────────────────────────────
    ("What is a melanocytic nevus?",
     "A melanocytic nevus (common mole) is a benign proliferation of melanocytes. Types: "
     "junctional (flat, at dermal-epidermal junction), compound (slightly raised, both "
     "junctional and dermal), intradermal (raised, dome-shaped, predominantly dermal). "
     "Most nevi are stable. Atypical (dysplastic) nevi have irregular features and "
     "increased melanoma risk."),

    ("When should a mole be removed?",
     "Mole removal is indicated when ABCDE criteria are present, the lesion has changed "
     "rapidly, bleeds or ulcerates spontaneously, dermoscopy reveals atypical features "
     "(irregular network, regression, atypical vessels), or clinician is uncertain. "
     "Any lesion suspicious for melanoma should be excised with 1-2mm margins for "
     "histopathological diagnosis rather than observed."),

    # ── Treatment + Risk ──────────────────────────────────────────────────
    ("What is Mohs micrographic surgery?",
     "Mohs surgery removes skin cancer in horizontal layers with immediate intraoperative "
     "microscopic margin assessment. Each layer is mapped; removal continues only where "
     "tumour remains. Achieves highest cure rates: 98-99% for BCC, 92-97% for SCC. "
     "Ideal for tumours on face, ears, eyelids, nose, and lips where tissue conservation "
     "is critical."),

    ("What is immunotherapy for skin cancer?",
     "Immune checkpoint inhibitors block PD-1 (pembrolizumab, nivolumab) or CTLA-4 "
     "(ipilimumab), restoring T-cell mediated tumour killing. Approved for: advanced "
     "melanoma (first-line and adjuvant), advanced SCC (cemiplimab), Merkel cell carcinoma. "
     "Combination ipilimumab + nivolumab achieves 5-year survival of ~52% in advanced "
     "melanoma. Immune-related adverse events can affect any organ system."),

    ("What factors increase skin cancer risk?",
     "Risk factors: cumulative UV radiation exposure (sunlight, tanning beds), fair "
     "Fitzpatrick skin phototype I-II, personal/family history of skin cancer, multiple "
     "nevi (>50) or dysplastic nevi, immunosuppression (transplant recipients have "
     "65-250× increased SCC risk), chronic scars or wounds, HPV infection, BRAF/NRAS "
     "mutations, xeroderma pigmentosum, and albinism."),

    ("What are high-risk features of squamous cell carcinoma?",
     "High-risk SCC features per NCCN guidelines: diameter >2cm, depth >2mm or Clark "
     "level IV/V, poor differentiation, perineural or lymphovascular invasion, location "
     "on ear or non-hair-bearing lip, arising in scar or chronic inflammation, "
     "immunosuppressed patient, and recurrent tumour. These features increase metastatic "
     "risk to 10-30% and require Mohs surgery, sentinel node biopsy consideration, and "
     "adjuvant radiation."),

    ("What is photodynamic therapy for skin lesions?",
     "PDT uses a photosensitising agent (aminolevulinic acid or methyl aminolevulinate) "
     "applied topically, preferentially absorbed by dysplastic cells. Activation with "
     "specific wavelength light (630nm red) generates reactive oxygen species that "
     "selectively destroy abnormal cells. Highly effective for AKs (70-90% clearance), "
     "superficial BCC, and Bowen's disease. Excellent cosmetic outcomes."),

    ("How is dermoscopy used in clinical practice?",
     "Dermoscopy uses polarised or immersion light to visualise subsurface structures "
     "invisible to the naked eye. Improves melanoma detection sensitivity from 71% to 90% "
     "and specificity from 81% to 90% vs naked-eye examination. Key structures: pigment "
     "network, aggregated globules, streaks/pseudopods, regression structures, milia-like "
     "cysts, and vascular patterns. Algorithms: ABCD rule, 7-point checklist, Menzies method."),

    ("What is the difference between ABCDE and ABCD dermoscopy?",
     "Clinical ABCDE rule (Asymmetry, Border, Color, Diameter, Evolution) is used by "
     "naked eye for patient self-examination and general screening. The dermoscopic ABCD "
     "rule (Asymmetry, Border, Color, Dermoscopic structures) is a quantitative scoring "
     "system producing a Total Dermoscopic Score. Both systems complement each other: "
     "ABCDE for initial detection, dermoscopic ABCD for quantitative risk stratification."),
]

out_dir = RAG_CORPUS_DIR / "medquad"
written = 0
for i, (q, a) in enumerate(DERM_QA):
    fname = f"derm_qa_{i+1:03d}.txt"
    (out_dir / fname).write_text(f"Q: {q}\nA: {a}\n", encoding="utf-8")
    written += 1

print(f"[OK] {written} dermatology QA pairs written to:")
print(f"     {out_dir}")
print()
print("Now check corpus for manual files:")
total_docs = len(list(RAG_CORPUS_DIR.rglob("*.txt")))
print(f"  Total documents: {total_docs}")
for sub in ["statpearls", "abcde_guidelines", "aad_guidelines", "dermnet", "medquad"]:
    cnt = len(list((RAG_CORPUS_DIR / sub).glob("*.txt")))
    tag = "[OK]" if cnt > 0 else "[EMPTY]"
    print(f"  {tag} {sub:<25} {cnt} files")

print()
print("Run Cell 4 (chunker) and Cell 5 (indexer) to load utilities,")
print("then Cell 6 to build the index.")


[OK] 26 dermatology QA pairs written to:
     /home/vjti-comp/Desktop/Final Project Code/VQA/rag_corpus/medquad

Now check corpus for manual files:
  Total documents: 26
  [EMPTY] statpearls                0 files
  [EMPTY] abcde_guidelines          0 files
  [EMPTY] aad_guidelines            0 files
  [EMPTY] dermnet                   0 files
  [OK] medquad                   26 files

Run Cell 4 (chunker) and Cell 5 (indexer) to load utilities,
then Cell 6 to build the index.


In [5]:
# Cell 4: Chunker Utility
# Splits .txt files into 350-word overlapping chunks.
# Source subfolder name becomes the "source" metadata tag in ChromaDB.

def chunk_text(text, source="", doc_id=""):
    text  = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    if not words: return []
    chunks = []; step = max(1, CHUNK_SIZE - CHUNK_OVERLAP)
    for i, start in enumerate(range(0, len(words), step)):
        ct = " ".join(words[start: start + CHUNK_SIZE])
        if len(ct) >= MIN_CHUNK_CHARS:
            chunks.append({"text": ct, "source": source,
                           "doc_id": doc_id, "chunk_index": i})
    return chunks

def chunk_corpus(corpus_dir=None):
    cdir = Path(corpus_dir or RAG_CORPUS_DIR)
    docs = [f for f in cdir.rglob("*.txt") if f.is_file()]
    if not docs:
        print(f"No documents in {cdir}")
        return []
    print(f"Chunking {len(docs)} documents...")
    all_chunks = []
    for path in sorted(docs):
        try:
            text = path.read_text(encoding="utf-8", errors="ignore")
            all_chunks.extend(
                chunk_text(text, source=path.parent.name, doc_id=path.stem))
        except Exception as e:
            print(f"  Warning: {path.name}: {e}")
    return all_chunks

print("[OK] Chunker ready.")


[OK] Chunker ready.


In [6]:
# Cell 5: Indexer Utility
# Embeds chunks with all-MiniLM-L6-v2 and stores in ChromaDB.
# First run downloads the model (~90 MB).

import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

_embed_model = None

def get_embed_model():
    global _embed_model
    if _embed_model is None:
        print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
        _embed_model = SentenceTransformer(EMBEDDING_MODEL)
        print("  Model loaded.")
    return _embed_model

def get_collection(reset=False):
    client = chromadb.PersistentClient(
        path=str(CHROMA_DB_DIR),
        settings=Settings(anonymized_telemetry=False))
    if reset:
        try:
            client.delete_collection(CHROMA_COLLECTION_NAME)
            print(f"  Deleted existing collection: {CHROMA_COLLECTION_NAME}")
        except Exception: pass
    return client.get_or_create_collection(
        name=CHROMA_COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"})

def build_index(chunks, reset=False):
    if not chunks:
        print("No chunks to index."); return 0
    model      = get_embed_model()
    collection = get_collection(reset=reset)
    existing   = collection.count()
    if existing > 0 and not reset:
        print(f"  Collection already has {existing:,} chunks.")
        print("  Set RESET=True in Cell 6 to wipe and rebuild.")
        return existing
    print(f"Indexing {len(chunks):,} chunks...")
    for start in tqdm(range(0, len(chunks), EMBED_BATCH_SIZE), desc="Embedding"):
        batch  = chunks[start: start + EMBED_BATCH_SIZE]
        texts  = [c["text"] for c in batch]
        embeds = model.encode(texts, normalize_embeddings=True).tolist()
        ids    = [f"{c['doc_id']}__{c['chunk_index']}__{uuid.uuid4().hex[:6]}"
                  for c in batch]
        metas  = [{"source": c["source"], "doc_id": c["doc_id"],
                   "chunk_index": c["chunk_index"]} for c in batch]
        collection.add(ids=ids, embeddings=embeds, documents=texts, metadatas=metas)
    total = collection.count()
    print(f"Done. {total:,} chunks in collection.")
    return total

def retrieve(question, top_k=3, source_filter=None):
    model      = get_embed_model()
    collection = get_collection()
    emb        = model.encode([question], normalize_embeddings=True).tolist()
    where      = {"source": source_filter} if source_filter else None
    res        = collection.query(
        query_embeddings=emb, n_results=top_k * 2, where=where)
    out = []
    for doc, meta, dist in zip(res["documents"][0],
                                res["metadatas"][0],
                                res["distances"][0]):
        sim = 1.0 - dist
        if sim >= SIMILARITY_THRESHOLD:
            out.append({"text": doc, "source": meta["source"],
                        "doc_id": meta["doc_id"], "score": round(sim, 4)})
        if len(out) >= top_k: break
    return out

print("[OK] Indexer + retriever ready.")


[OK] Indexer + retriever ready.


In [7]:
# Cell 6: Build RAG Index  ← MAIN CELL
# Set RESET=True to wipe existing index and rebuild from scratch.

RESET = False

print("=" * 60)
print(" RAG Ingestion Pipeline")
print("=" * 60)
print(f" Corpus  : {RAG_CORPUS_DIR}")
print(f" ChromaDB: {CHROMA_DB_DIR}")
print(f" Reset   : {RESET}")
print()

# Chunk all documents
all_chunks = chunk_corpus()

if not all_chunks:
    print("No documents found.")
    print("Make sure Cell 3 has been run (writes medquad/ QA pairs).")
    print("Then add .txt files for statpearls/ aad_guidelines/ dermnet/ abcde_guidelines/")
else:
    # Source breakdown
    src_counts = Counter(c["source"] for c in all_chunks)
    print(f"Generated {len(all_chunks):,} chunks total:")
    for src, cnt in sorted(src_counts.items(), key=lambda x: -x[1]):
        print(f"  {src:<25} {cnt:>6,} chunks")
    print()

    # Build index
    total = build_index(all_chunks, reset=RESET)

    print()
    print("=" * 60)
    print(f" Index built: {total:,} chunks in ChromaDB")
    print(f" Location   : {CHROMA_DB_DIR}")
    print(" Run Cell 7 to test retrieval.")
    print("=" * 60)


 RAG Ingestion Pipeline
 Corpus  : /home/vjti-comp/Desktop/Final Project Code/VQA/rag_corpus
 ChromaDB: /home/vjti-comp/Desktop/Final Project Code/VQA/chroma_db
 Reset   : False

Chunking 26 documents...
Generated 26 chunks total:
  medquad                       26 chunks

Loading embedding model: all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  Model loaded.
Indexing 26 chunks...


Embedding: 100%|██████████| 1/1 [00:00<00:00,  3.53it/s]

Done. 26 chunks in collection.

 Index built: 26 chunks in ChromaDB
 Location   : /home/vjti-comp/Desktop/Final Project Code/VQA/chroma_db
 Run Cell 7 to test retrieval.


In [8]:
# Cell 7: Test Retrieval
# Tests all 4 question router categories + unfiltered queries
# Covers all 6 disease classes

tests = [
    # (question, source_filter, expected_category)
    # ── diagnosis / dermnet ───────────────────────────────────────────
    ("What is melanoma?",                                "dermnet",          "diagnosis"),
    ("What does basal cell carcinoma look like?",        "dermnet",          "diagnosis"),
    ("How is seborrheic keratosis identified?",          "dermnet",          "diagnosis"),

    # ── ABCDE / abcde_guidelines ─────────────────────────────────────
    ("What are the ABCDE criteria?",                     "abcde_guidelines", "abcde"),
    ("What does asymmetry score mean in dermoscopy?",    None,               "abcde"),
    ("What color variations indicate malignancy?",       None,               "abcde"),

    # ── risk / aad_guidelines ────────────────────────────────────────
    ("What factors increase skin cancer risk?",          "aad_guidelines",   "risk"),
    ("How dangerous is squamous cell carcinoma?",        None,               "risk"),

    # ── treatment / aad_guidelines ───────────────────────────────────
    ("How is melanoma treated?",                         "aad_guidelines",   "treatment"),
    ("What surgical margins are used for melanoma?",     None,               "treatment"),
    ("What is Mohs micrographic surgery?",               None,               "treatment"),

    # ── general / no filter ──────────────────────────────────────────
    ("What is actinic keratosis?",                       None,               "general"),
    ("Difference between ABCDE and ABCD dermoscopy?",   None,               "general"),
    ("When should a mole be removed?",                   None,               "general"),
]

print("=" * 70)
print(f" Testing {len(tests)} queries across all router categories")
print("=" * 70)

hits = 0
by_category = {}
for q, filt, cat in tests:
    results = retrieve(q, top_k=2, source_filter=filt)
    ok      = bool(results)
    if ok: hits += 1
    by_category.setdefault(cat, []).append(ok)

    tag = "OK  " if ok else "MISS"
    print(f"\n[{tag}] [{cat}] {q}")
    if filt: print(f"       filter={filt}")
    if results:
        for r in results:
            print(f"  [{r['score']:.3f}] [{r['source']}] {r['text'][:100]}...")
    else:
        print("  No results above threshold — add more corpus files for this category")

print()
print("=" * 70)
print(f" Results: {hits}/{len(tests)} queries returned results")
print()
print(" By category:")
for cat, results in by_category.items():
    n_ok  = sum(results)
    total = len(results)
    bar   = "█" * n_ok + "░" * (total - n_ok)
    print(f"   {cat:<20} {n_ok}/{total}  {bar}")

if hits < len(tests):
    missing = len(tests) - hits
    print()
    print(f" {missing} queries missed — add .txt files to the corresponding corpus folders")
    print(" to improve coverage. Re-run Cell 6 with RESET=True after adding files.")
print("=" * 70)


 Testing 14 queries across all router categories

[MISS] [diagnosis] What is melanoma?
       filter=dermnet
  No results above threshold — add more corpus files for this category

[MISS] [diagnosis] What does basal cell carcinoma look like?
       filter=dermnet
  No results above threshold — add more corpus files for this category

[MISS] [diagnosis] How is seborrheic keratosis identified?
       filter=dermnet
  No results above threshold — add more corpus files for this category

[MISS] [abcde] What are the ABCDE criteria?
       filter=abcde_guidelines
  No results above threshold — add more corpus files for this category

[OK  ] [abcde] What does asymmetry score mean in dermoscopy?
  [0.848] [medquad] Q: What does asymmetry mean in dermoscopy? A: Asymmetry is evaluated by bisecting the lesion along t...
  [0.588] [medquad] Q: What is the difference between ABCDE and ABCD dermoscopy? A: Clinical ABCDE rule (Asymmetry, Bord...

[OK  ] [abcde] What color variations indicate malignan

In [9]:
# Cell 8: Stats + Bot Readiness Check
import chromadb
from chromadb.config import Settings

client = chromadb.PersistentClient(
    path=str(CHROMA_DB_DIR),
    settings=Settings(anonymized_telemetry=False))

try:
    col   = client.get_collection(CHROMA_COLLECTION_NAME)
    total = col.count()
except Exception as e:
    print(f"Collection not found: {e}")
    print("Run Cell 6 first."); raise

print("=" * 60)
print(" RAG Index Summary")
print("=" * 60)
print(f"  Chunks        : {total:,}")
print(f"  Collection    : {CHROMA_COLLECTION_NAME}")
print(f"  ChromaDB      : {CHROMA_DB_DIR}")
print(f"  Embedding     : {EMBEDDING_MODEL}")

# Source breakdown with bar chart
res  = col.get(limit=min(total, 100000), include=["metadatas"])
srcs = Counter(m["source"] for m in res["metadatas"])
print(f"\n  Sources ({len(srcs)}):")
max_cnt = max(srcs.values()) if srcs else 1
for src, cnt in sorted(srcs.items(), key=lambda x: -x[1]):
    bar = "█" * (cnt * 30 // max_cnt)
    print(f"    {src:<25} {cnt:>5,}  {bar}")

# Bot file check
print(f"\n  Bot files (in {PROJECT_DIR}):")
bot_files = [
    ("rag_config.py",               True,  "required"),
    ("rag_retriever.py",            True,  "required"),
    ("question_router.py",          True,  "required"),
    ("llama_reasoner.py",           True,  "required"),
    ("dermatology_orchestrator.py", True,  "required"),
    ("dermatology_bot.py",          True,  "required — FastAPI server"),
]
all_ok = True
for fname, required, note in bot_files:
    exists = (PROJECT_DIR / fname).exists()
    tag    = "[OK]    " if exists else "[MISSING]"
    if not exists: all_ok = False
    print(f"    {tag} {fname:<35} {note}")

# Corpus completeness
print(f"\n  Corpus folders:")
for sub in ["statpearls","abcde_guidelines","aad_guidelines","dermnet","medquad"]:
    cnt    = len(list((RAG_CORPUS_DIR / sub).glob("*.txt")))
    kb     = sum(f.stat().st_size for f in (RAG_CORPUS_DIR / sub).glob("*.txt")) // 1024
    tag    = "[OK]" if cnt > 0 else "[EMPTY]"
    print(f"    {tag} {sub:<25} {cnt:>3} files  {kb:>5} KB")

print()
if total >= 50 and all_ok:
    print("  READY. Start the bot:")
    print(f"    cd \"{PROJECT_DIR}\"")
    print("    python dermatology_bot.py")
elif total == 0:
    print("  Index is empty — run Cells 3→4→5→6.")
elif not all_ok:
    print("  Bot files missing — check paths above.")
else:
    print(f"  Index has {total} chunks (low). Add more corpus files + rebuild.")
print("=" * 60)


 RAG Index Summary
  Chunks        : 26
  Collection    : dermatology_kb
  ChromaDB      : /home/vjti-comp/Desktop/Final Project Code/VQA/chroma_db
  Embedding     : all-MiniLM-L6-v2

  Sources (1):
    medquad                      26  ██████████████████████████████

  Bot files (in /home/vjti-comp/Desktop/Final Project Code/VQA):
    [OK]     rag_config.py                       required
    [OK]     rag_retriever.py                    required
    [OK]     question_router.py                  required
    [OK]     llama_reasoner.py                   required
    [OK]     dermatology_orchestrator.py         required
    [OK]     dermatology_bot.py                  required — FastAPI server

  Corpus folders:
    [EMPTY] statpearls                  0 files      0 KB
    [EMPTY] abcde_guidelines            0 files      0 KB
    [EMPTY] aad_guidelines              0 files      0 KB
    [EMPTY] dermnet                     0 files      0 KB
    [OK] medquad                    26 files     10

In [10]:
# Cell 9: Update rag_config.py with correct RTX 4070 paths
rag_config_path = PROJECT_DIR / "rag_config.py"

config_content = f'''"""
Stage 7 RAG configuration — RTX 4070 machine.
All RAG-related scripts import from here.
"""
import os
from pathlib import Path

# Corpus documents
RAG_CORPUS_DIR = Path(os.environ.get(
    "STAGE7_RAG_CORPUS_DIR",
    "{RAG_CORPUS_DIR}"))

# ChromaDB persistent vector store
CHROMA_DB_DIR = Path(os.environ.get(
    "STAGE7_CHROMA_DIR",
    "{CHROMA_DB_DIR}"))

CHROMA_COLLECTION_NAME = "dermatology_kb"
EMBEDDING_MODEL        = "all-MiniLM-L6-v2"

CHUNK_SIZE           = 350
CHUNK_OVERLAP        = 60
MIN_CHUNK_CHARS      = 40
TOP_K                = 3
SIMILARITY_THRESHOLD = 0.25
EMBED_BATCH_SIZE     = 64
'''

rag_config_path.write_text(config_content, encoding="utf-8")
print(f"[OK] Updated: {rag_config_path}")
print()
print("Contents:")
print(config_content)


[OK] Updated: /home/vjti-comp/Desktop/Final Project Code/VQA/rag_config.py

Contents:
"""
Stage 7 RAG configuration — RTX 4070 machine.
All RAG-related scripts import from here.
"""
import os
from pathlib import Path

# Corpus documents
RAG_CORPUS_DIR = Path(os.environ.get(
    "STAGE7_RAG_CORPUS_DIR",
    "/home/vjti-comp/Desktop/Final Project Code/VQA/rag_corpus"))

# ChromaDB persistent vector store
CHROMA_DB_DIR = Path(os.environ.get(
    "STAGE7_CHROMA_DIR",
    "/home/vjti-comp/Desktop/Final Project Code/VQA/chroma_db"))

CHROMA_COLLECTION_NAME = "dermatology_kb"
EMBEDDING_MODEL        = "all-MiniLM-L6-v2"

CHUNK_SIZE           = 350
CHUNK_OVERLAP        = 60
MIN_CHUNK_CHARS      = 40
TOP_K                = 3
SIMILARITY_THRESHOLD = 0.25
EMBED_BATCH_SIZE     = 64

